# Project 04 — First Machine Learning Classifier

**Goal:** predict the recorded Titanic survival outcome with a reproducible train/test split and a scikit-learn decision tree.

The preprocessing pipeline imputes missing values and one-hot encodes categorical fields using training data only. The test set is held out until evaluation. Results are a learning demonstration, not a causal or operational model.

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

candidates = [Path("../data/titanic.csv"), Path("data/titanic.csv"), Path("../../data/titanic.csv")]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Place the included titanic.csv in the portfolio data folder.")
df = pd.read_csv(DATA_PATH)
FEATURES = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
X = df[FEATURES]
y = df["survived"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training rows: {len(X_train)} | held-out test rows: {len(X_test)}")
display(X_train.head())

Training rows: 712 | held-out test rows: 179
     pclass     sex   age  sibsp  parch      fare embarked
692       3    male   NaN      0      0   56.4958        S
481       2    male   NaN      0      0    0.0000        S
527       1    male   NaN      0      0  221.7792        S
855       3  female  18.0      0      1    9.3500        S
801       2  female  31.0      1      1   26.2500        S


## 1. Build the preprocessing and decision-tree pipeline

In [2]:
numeric_features = ["pclass", "age", "sibsp", "parch", "fare"]
categorical_features = ["sex", "embarked"]
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)),
])
model.fit(X_train, y_train)

## 2. Evaluate once on the held-out test split

In [3]:
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]
accuracy = float(accuracy_score(y_test, y_pred))
probability_mse = float(mean_squared_error(y_test, y_probability))
majority_class = int(y_train.mode().iloc[0])
baseline_accuracy = float(accuracy_score(y_test, [majority_class] * len(y_test)))
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
report = classification_report(y_test, y_pred, target_names=["Did not survive", "Survived"], output_dict=True, zero_division=0)
results = {
    "dataset_rows": int(len(df)), "train_rows": int(len(X_train)), "test_rows": int(len(X_test)),
    "random_state": 42, "accuracy": accuracy, "majority_baseline_accuracy": baseline_accuracy,
    "probability_mse_brier_score": probability_mse, "confusion_matrix_labels": ["did_not_survive", "survived"],
    "confusion_matrix": cm.tolist(), "classification_report": report,
}
print(f"Accuracy: {accuracy:.3f}")
print(f"Majority-class baseline accuracy: {baseline_accuracy:.3f}")
print(f"Probability MSE (Brier score): {probability_mse:.3f}")
print("Confusion matrix (rows=true, columns=predicted; 0=not survived, 1=survived):")
print(cm)
display(pd.DataFrame(report).T.round(3))

OUT = Path(".")
(OUT / "model").mkdir(exist_ok=True)
(OUT / "charts").mkdir(exist_ok=True)
with open(OUT / "metrics.json", "w", encoding="utf-8") as stream:
    json.dump(results, stream, indent=2)
joblib.dump(model, OUT / "model" / "titanic_decision_tree_pipeline.joblib")
fig, ax = plt.subplots(figsize=(5.8, 4.8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, square=True,
            xticklabels=["Not survived", "Survived"], yticklabels=["Not survived", "Survived"], ax=ax)
ax.set(title="Held-out test confusion matrix", xlabel="Predicted label", ylabel="True label")
fig.tight_layout()
fig.savefig(OUT / "charts" / "confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

Accuracy: 0.777
Majority-class baseline accuracy: 0.615
Probability MSE (Brier score): 0.166
Confusion matrix (rows=true, columns=predicted; 0=not survived, 1=survived):
[[101   9]
 [ 31  38]]
                 precision  recall  f1-score  support
Did not survive      0.765   0.918     0.835  110.000
Survived             0.809   0.551     0.655   69.000
accuracy             0.777   0.777     0.777    0.777
macro avg            0.787   0.734     0.745  179.000
weighted avg         0.782   0.777     0.766  179.000
C:\Users\gana6\Documents\Codex\2026-11-12\referenced-chatgpt-conversation-this-is-an\outputs\InternCircle_Data_Science_Projects\project_04_classifier\titanic_classifier.ipynb:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ],


## Limitations

- This is a small, historical, single-split teaching example. A different split can produce different scores.
- The dataset is observational and incomplete; feature associations do not explain causes.
- The result may reflect historical rescue and social structures. Do not use it for real-world decisions about people.
- Accuracy and the confusion matrix should be read together; the probability MSE summarizes errors in the predicted survival probabilities.